In [ ]:
# ! pip install pandas
# ! pip install regex
# ! pip install pymongo
# ! pip install dotenv
# ! pip install matplotlib

In [ ]:
import pandas as pd
import sys
import os
import shutil

In [ ]:
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend (no GUI)

import matplotlib.pyplot as plt
plt.ioff()  # Turn off interactive mode
plt.show = lambda *args, **kwargs: None 

In [ ]:
bigcodebench = pd.read_csv("/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/datasets/open_ended_format/bigcodebench_test.csv", header = 0, encoding='utf-8')
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [ ]:
from code_generation.utility.humaneval_helper import CodeGenerationHumanEvalHelper
from code_generation.utility.bigcodebench_helper import CodeGenerationBigCodeBenchHelper
from database import MongoDBHelper

In [ ]:
sample_qn = bigcodebench.iloc[0]
print(bigcodebench.columns.values)
prompt = sample_qn['complete_prompt']
canonical_solution = sample_qn['canonical_solution'].replace("\"", '"""').replace("\'", "'''")
instruct_prompt = sample_qn['instruct_prompt']
test = sample_qn['test'].replace("\"", '"""').replace("\'", "'''")
# original_test = sample_qn['test']
# print(prompt)

In [ ]:
prompt, qn_desc = CodeGenerationBigCodeBenchHelper.seperate_original_desciptions(prompt = sample_qn['complete_prompt'])

In [ ]:
task_doc_string, example = CodeGenerationBigCodeBenchHelper.extract_examples(qn_desc)

In [ ]:
natural_language_instruct, code_instruct= CodeGenerationBigCodeBenchHelper.split_code_from_instruct_prompt(instruct_prompt=instruct_prompt)

In [ ]:
full_sol = CodeGenerationBigCodeBenchHelper.obtain_full_sol(
canonical_sol=canonical_solution, 
code_instruct=code_instruct,
test = test)

# Connecting to MongoDB

In [ ]:
mongodbHelper = MongoDBHelper()
mongodbHelper.check_database_connectivity()

In [ ]:
print(curr_dir)
os.listdir(curr_dir)

In [ ]:
db = mongodbHelper.client["Base_Questions_DB"]
open_ended_db = db["BigCodeBench_Code_Generation"]

In [ ]:
# %%script false --no-raise-error

to_check_example = []
to_check_test = []

try: 
    for idx in range(
        bigcodebench.__len__(),
        ):

        qn_id = f"BigCodeBencho{idx-len(to_check_example)-len(to_check_test)}"

        qn_details = open_ended_db.find_one({"_id" : qn_id})        # checking if this qn_id already exists in the db

        qn = bigcodebench.iloc[idx]

        original_prompt = qn['complete_prompt'].replace(r"\\x", r"\\\\x")
        canonical_solution = qn['canonical_solution']
        instruct_prompt = qn['instruct_prompt']

        test = qn['test']
        original_test_id = qn['task_id']

        # storing all the file names in the current directory as a snapshot
        # this is a necessary step to remove any new files created from running the tasks
        snapshot_dir = os.listdir(curr_dir)

        prompt, qn_desc = CodeGenerationBigCodeBenchHelper.seperate_original_desciptions(original_prompt)

        task_doc_string, example = CodeGenerationBigCodeBenchHelper.extract_examples(qn_desc)

        natural_language_instruct, code_instruct= CodeGenerationBigCodeBenchHelper.split_code_from_instruct_prompt(instruct_prompt=instruct_prompt)
        try: 
            full_sol = CodeGenerationBigCodeBenchHelper.obtain_full_sol(
                canonical_sol=canonical_solution, 
                code_instruct=code_instruct,
                test = test)
        except Exception as e:
            print(e)
            print(original_test_id)
            to_check_test.append(original_test_id)
            continue
        
        entry_dict = {
            "_id" : qn_id,
            "qn" : code_instruct,
            "qn_desc": task_doc_string,
            "canon_solution" : canonical_solution,
            "original_qn_desc" : qn_desc,
            "examples": example,
            "check" : test,
            "original_id": original_test_id
        }

        curr_dir_snapshot = os.listdir(curr_dir)
        for file_name in curr_dir_snapshot:
            if file_name not in snapshot_dir:
                file_path = os.path.join(curr_dir, file_name)
                if os.path.isdir(file_path):
                    shutil.rmtree(file_path)
                else:
                    os.remove(file_path)


        if qn_details is None:
            open_ended_db.insert_one(entry_dict)
            print('Added entry to database: {id}'.format(id = qn_id))
        else:
            open_ended_db.update_one({"_id" : qn_id}, update = {"$set": entry_dict})
            print('Updated existing entry in database: {id}'.format(id = qn_id))
except KeyboardInterrupt:
    print(original_test_id)
except Exception as e:
    print(original_test_id)
    print(e)
    print(to_check_test)


print(to_check_example if len(to_check_example) > 0 else "All cases contains examples. Nothing to check!")
print(to_check_test if len(to_check_test) > 0 else "All test cases passed. Nothing to check!")

In [ ]:
%%script false --no-raise-error
samples = {'BigCodeBench/871', 'BigCodeBench/940', 'BigCodeBench/362', 'BigCodeBench/1022', 'BigCodeBench/779', 'BigCodeBench/1049', 'BigCodeBench/83', 'BigCodeBench/348', 'BigCodeBench/131', 'BigCodeBench/724', 'BigCodeBench/80', 'BigCodeBench/372', 'BigCodeBench/158', 'BigCodeBench/498', 'BigCodeBench/723', 'BigCodeBench/1084', 'BigCodeBench/1129', 'BigCodeBench/808', 'BigCodeBench/82', 'BigCodeBench/656', 'BigCodeBench/632', 'BigCodeBench/806', 'BigCodeBench/274', 'BigCodeBench/130', 'BigCodeBench/227', 'BigCodeBench/245', 'BigCodeBench/346', 'BigCodeBench/847', 'BigCodeBench/596', 'BigCodeBench/370', 'BigCodeBench/495', 'BigCodeBench/501', 'BigCodeBench/972', 'BigCodeBench/634', 'BigCodeBench/1109', 'BigCodeBench/987', 'BigCodeBench/202', 'BigCodeBench/358', 'BigCodeBench/205', 'BigCodeBench/964', 'BigCodeBench/683', 'BigCodeBench/270', 'BigCodeBench/926', 'BigCodeBench/490', 'BigCodeBench/565', 'BigCodeBench/203', 'BigCodeBench/985', 'BigCodeBench/39', 'BigCodeBench/655', 'BigCodeBench/924', 'BigCodeBench/708', 'BigCodeBench/360', 'BigCodeBench/374', 'BigCodeBench/81', 'BigCodeBench/593', 'BigCodeBench/746', 'BigCodeBench/383', 'BigCodeBench/726', 'BigCodeBench/728', 'BigCodeBench/215', 'BigCodeBench/101', 'BigCodeBench/334', 'BigCodeBench/734', 'BigCodeBench/629', 'BigCodeBench/461', 'BigCodeBench/115', 'BigCodeBench/132', 'BigCodeBench/812', 'BigCodeBench/1020', 'BigCodeBench/364', 'BigCodeBench/361', 'BigCodeBench/177', 'BigCodeBench/657', 'BigCodeBench/927', 'BigCodeBench/220', 'BigCodeBench/736', 'BigCodeBench/1028', 'BigCodeBench/658', 'BigCodeBench/363', 'BigCodeBench/804', 'BigCodeBench/986', 'BigCodeBench/1009', 'BigCodeBench/686', 'BigCodeBench/1008', 'BigCodeBench/761', 'BigCodeBench/1095', 'BigCodeBench/1124', 'BigCodeBench/458', 'BigCodeBench/412', 'BigCodeBench/844', 'BigCodeBench/612', 'BigCodeBench/849', 'BigCodeBench/192', 'BigCodeBench/994', 'BigCodeBench/1000', 'BigCodeBench/577', 'BigCodeBench/867'}

to_check_example = []
to_check_test = set()

for qn_id in samples:
    try: 

        print(qn_id)
        idx = int(qn_id.split('BigCodeBench/')[-1])

        qn_id = f"BigCodeBencho{idx-len(to_check_example)-len(samples)}"

        qn_details = open_ended_db.find_one({"_id" : qn_id})        # checking if this qn_id already exists in the db

        qn = bigcodebench.iloc[idx]
        original_prompt = qn['complete_prompt']
        canonical_solution = qn['canonical_solution'].replace("\"", '"""').replace("\'", "'''")
        instruct_prompt = qn['instruct_prompt']
        test = qn['test'].replace("\"", '"""').replace("\'", "'''")
        original_test_id = qn['task_id']

        prompt, qn_desc = CodeGenerationBigCodeBenchHelper.seperate_original_desciptions(original_prompt)

        task_doc_string, example = CodeGenerationBigCodeBenchHelper.extract_examples(qn_desc)

        natural_language_instruct, code_instruct= CodeGenerationBigCodeBenchHelper.split_code_from_instruct_prompt(instruct_prompt=instruct_prompt)
        try: 
            full_sol = CodeGenerationBigCodeBenchHelper.obtain_full_sol(
                canonical_sol=canonical_solution, 
                code_instruct=code_instruct,
                test = test)
        except Exception as e:
            print(e)
            to_check_test.add(original_test_id)
            continue
        
        entry_dict = {
            "_id" : qn_id,
            "qn" : code_instruct,
            "task_doc_string": task_doc_string,
            "canon_solution" : canonical_solution,
            "qn_desc" : qn_desc,
            "example": example,
            "check" : test,
            "original_id": original_test_id
        }
        if qn_details is None:
            open_ended_db.insert_one(entry_dict)
            print('Added entry to database: {id}'.format(id = qn_id))
        else:
            open_ended_db.update_one({"_id" : qn_id}, update = {"$set": entry_dict})
            print('Updated existing entry in database: {id}'.format(id = qn_id))
    except KeyboardInterrupt:
        print(original_test_id)
    except Exception as e:
        print(original_test_id)
        print(e)
        continue


print(to_check_example if len(to_check_example) > 0 else "All cases contains examples. Nothing to check!")
print(to_check_test if len(to_check_test) > 0 else "All test cases passed. Nothing to check!")

In [ ]:
p = {'BigCodeBench/200', 'BigCodeBench/960', 'BigCodeBench/186', 'BigCodeBench/484', 'BigCodeBench/579', 'BigCodeBench/o37', 'BigCodeBench/652', 'BigCodeBench/764', 'BigCodeBench/900', 'BigCodeBench/929', 'BigCodeBench/342', 'BigCodeBench/448', 'BigCodeBench/794', 'BigCodeBench/423', 'BigCodeBench/320', 'BigCodeBench/997', 'BigCodeBench/888', 'BigCodeBench/669', 'BigCodeBench/774', 'BigCodeBench/616', 'BigCodeBench/853', 'BigCodeBench/989', 'BigCodeBench/956', 'BigCodeBench/156', 'BigCodeBench/309', 'BigCodeBench/162', 'BigCodeBench/356', 'BigCodeBench/847', 'BigCodeBench/o75', 'BigCodeBench/659', 'BigCodeBench/308', 'BigCodeBench/248', 'BigCodeBench/881', 'BigCodeBench/o28', 'BigCodeBench/249', 'BigCodeBench/703', 'BigCodeBench/459', 'BigCodeBench/773', 'BigCodeBench/341', 'BigCodeBench/532', 'BigCodeBench/627', 'BigCodeBench/016', 'BigCodeBench/734', 'BigCodeBench/334', 'BigCodeBench/846', 'BigCodeBench/657', 'BigCodeBench/996', 'BigCodeBench/949', 'BigCodeBench/256', 'BigCodeBench/938', 'BigCodeBench/148', 'BigCodeBench/o89', 'BigCodeBench/185', 'BigCodeBench/553', 'BigCodeBench/561', 'BigCodeBench/o15', 'BigCodeBench/670', 'BigCodeBench/030', 'BigCodeBench/119', 'BigCodeBench/o50', 'BigCodeBench/874', 'BigCodeBench/o93', 'BigCodeBench/912', 'BigCodeBench/590', 'BigCodeBench/224', 'BigCodeBench/580', 'BigCodeBench/104', 'BigCodeBench/531', 'BigCodeBench/o76', 'BigCodeBench/545', 'BigCodeBench/547', 'BigCodeBench/o52', 'BigCodeBench/422', 'BigCodeBench/947', 'BigCodeBench/910', 'BigCodeBench/520', 'BigCodeBench/615', 'BigCodeBench/710', 'BigCodeBench/648', 'BigCodeBench/930', 'BigCodeBench/859', 'BigCodeBench/175', 'BigCodeBench/105', 'BigCodeBench/771', 'BigCodeBench/120', 'BigCodeBench/455', 'BigCodeBench/340', 'BigCodeBench/458', 'BigCodeBench/174', 'BigCodeBench/008', 'BigCodeBench/350', 'BigCodeBench/601', 'BigCodeBench/577', 'BigCodeBench/005', 'BigCodeBench/701', 'BigCodeBench/466', 'BigCodeBench/004', 'BigCodeBench/021', 'BigCodeBench/775', 'BigCodeBench/741', 'BigCodeBench/664', 'BigCodeBench/250', 'BigCodeBench/995', 'BigCodeBench/398', 'BigCodeBench/369'}
print(len(p))

x = ['BigCodeBench/80', 'BigCodeBench/81', 'BigCodeBench/82', 'BigCodeBench/83', 'BigCodeBench/101', 'BigCodeBench/115', 'BigCodeBench/177', 'BigCodeBench/205', 'BigCodeBench/220', 'BigCodeBench/245', 'BigCodeBench/334', 'BigCodeBench/363', 'BigCodeBench/372', 'BigCodeBench/383', 'BigCodeBench/501', 'BigCodeBench/593', 'BigCodeBench/596', 'BigCodeBench/612', 'BigCodeBench/634', 'BigCodeBench/683', 'BigCodeBench/686', 'BigCodeBench/734', 'BigCodeBench/736', 'BigCodeBench/779', 'BigCodeBench/940', 'BigCodeBench/964', 'BigCodeBench/1028', 'BigCodeBench/1109']
print(len(x))

c1 = 0
for i in p:
    if i not in x:
        print(f"Failed in p: {i}")
        c1+= 1
c2 = 0
for i in x:
    if i not in p:
        print(f'failed in x: {i}')
        c2 += 1

print(c1, c2)